# Dự báo số lượng đơn hàng - Trà sữa Mây Nhớ

Notebook này tạo dữ liệu mô phỏng từ 01/2020 đến 09/2026 và huấn luyện mô hình SARIMAX với biến ngoại sinh:
- `Y_t`: số đơn hàng mỗi ngày
- `X1`: nhiệt độ trung bình ngày, phân phối Normal và có tính mùa vụ
- `X2`: mã giảm giá, phân phối Binomial với xác suất 0.4

> Nếu có dữ liệu thực tế, thay phần tạo dữ liệu bằng lệnh `pd.read_csv(...)`.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn statsmodels scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Tạo dữ liệu từ 01/2020 đến hết 09/2026
dates = pd.date_range('2020-01-01', '2026-09-30', freq='D')
n = len(dates)
t = np.arange(n)

# X1: nhiệt độ Normal, có mùa vụ theo năm
temperature = 27 + 5 * np.sin(2 * np.pi * t / 365.25) + np.random.normal(0, 2.5, n)
temperature = np.clip(temperature, 15, 38)

# X2: mã giảm giá Binomial(n=1, p=0.4)
discount = np.random.binomial(1, 0.4, n)

# Y: số đơn hàng mô phỏng
weekend_effect = np.where(dates.dayofweek >= 5, 10, 0)
trend = 0.01 * t
noise = np.random.normal(0, 7, n)
orders = (45 + 2.2 * temperature + 18 * discount + weekend_effect + trend + noise)
orders = np.maximum(np.round(orders), 1).astype(int)

df = pd.DataFrame({
    'orders': orders,
    'temperature': temperature,
    'discount': discount
}, index=dates)
df.index.name = 'date'
df.head()

In [ ]:
print(df.describe())
print(f'Khoảng thời gian: {df.index.min().date()} đến {df.index.max().date()}')
print(f'Tỷ lệ ngày có mã giảm giá: {df.discount.mean():.2%}')

fig, ax = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
df.orders.plot(ax=ax[0], title='Số đơn hàng theo ngày', color='tab:blue')
df.temperature.plot(ax=ax[1], title='Nhiệt độ trung bình (°C)', color='tab:orange')
df.discount.plot(ax=ax[2], title='Mã giảm giá (0/1)', color='tab:green')
plt.tight_layout()
plt.show()

In [ ]:
# Chia train/test: 30 ngày cuối làm tập kiểm tra
test_days = 30
train = df.iloc[:-test_days].copy()
test = df.iloc[-test_days:].copy()

# SARIMAX hỗ trợ Y, X1, X2; seasonal_order=(P,D,Q,7) để mô hình hóa mùa vụ tuần
model = SARIMAX(
    train['orders'],
    exog=train[['temperature', 'discount']],
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(model.summary())

In [ ]:
# Đánh giá trên 30 ngày cuối
pred = model.get_forecast(steps=test_days, exog=test[['temperature', 'discount']]).predicted_mean
pred = pd.Series(np.maximum(pred, 0), index=test.index)

mae = mean_absolute_error(test['orders'], pred)
rmse = np.sqrt(mean_squared_error(test['orders'], pred))
mape = np.mean(np.abs((test['orders'] - pred) / test['orders'])) * 100
print(f'MAE:  {mae:.2f} đơn')
print(f'RMSE: {rmse:.2f} đơn')
print(f'MAPE: {mape:.2f}%')

plt.figure(figsize=(15, 5))
plt.plot(train.index[-90:], train.orders.tail(90), label='Thực tế - train')
plt.plot(test.index, test.orders, label='Thực tế - test', linewidth=2)
plt.plot(test.index, pred, label='Dự báo', linewidth=2)
plt.title('Đánh giá dự báo 30 ngày cuối')
plt.legend()
plt.show()

In [ ]:
# Dự báo 30 ngày tiếp theo
future_dates = pd.date_range(df.index[-1] + pd.Timedelta(days=1), periods=30, freq='D')
future_temperature = 27 + 5 * np.sin(2 * np.pi * (np.arange(n, n + 30)) / 365.25)
future_discount = np.random.binomial(1, 0.4, 30)
future_exog = pd.DataFrame({
    'temperature': future_temperature,
    'discount': future_discount
}, index=future_dates)

final_model = SARIMAX(
    df['orders'], exog=df[['temperature', 'discount']],
    order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)
forecast = final_model.get_forecast(steps=30, exog=future_exog)
forecast_df = future_exog.copy()
forecast_df['predicted_orders'] = np.maximum(np.round(forecast.predicted_mean), 0).astype(int)
forecast_df['lower_ci'] = np.maximum(np.round(forecast.conf_int().iloc[:, 0]), 0).astype(int)
forecast_df['upper_ci'] = np.maximum(np.round(forecast.conf_int().iloc[:, 1]), 0).astype(int)
forecast_df

In [ ]:
# Lưu dữ liệu và kết quả
df.to_csv('bubble_tea_orders.csv')
forecast_df.to_csv('forecast_30_days.csv')

plt.figure(figsize=(15, 5))
plt.plot(df.index[-90:], df.orders.tail(90), label='90 ngày gần nhất')
plt.plot(forecast_df.index, forecast_df.predicted_orders, label='Dự báo 30 ngày', color='red')
plt.fill_between(forecast_df.index, forecast_df.lower_ci, forecast_df.upper_ci, alpha=.2, color='red', label='Khoảng tin cậy')
plt.title('Dự báo đơn hàng Trà sữa Mây Nhớ')
plt.legend()
plt.show()